# NLP-Based Smart FAQ Retrieval and Question Answering System

**CSE 4122 — Natural Language Processing Laboratory, Phase 1**

This notebook walks through the whole system end to end: how the FAQ corpora are
loaded, how text is preprocessed, how the TF-IDF index is built, how a query is
matched by cosine similarity, how the rejection threshold is chosen, and how the
final results were measured.

The system **retrieves an existing answer**. It does not generate answers.

The same engine runs over two independent corpora — University and E-commerce —
each with its own preprocessing configuration and its own threshold, both chosen
from validation data only.

## 1. Setup

Everything below reads local CSV files. No internet connection is used.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# Run from the project root so `src` is importable.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.data_loader import (
    discover_corpora,
    load_corpus_config,
    load_faq_dataset,
    load_query_dataset,
)
from src.preprocessing import preprocess_text
from src.tfidf_retrieval import answer_query, build_tfidf_index, retrieve_tfidf
from src.evaluation import PREPROCESSING_CONFIGS, evaluate_tfidf, tune_threshold

pd.set_option("display.max_colwidth", 90)
print("Project root:", ROOT)

## 2. Discovering and loading the corpora

`discover_corpora` finds any directory under `data/` that contains a
`faq_dataset.csv`. Adding a third corpus needs no code change — this is what
"corpus-configurable" means in practice.

`load_faq_dataset` validates the schema on the way in: integer ids, no duplicate
ids, no blank fields, and no duplicate questions (exact or normalized).

In [ ]:
corpora = discover_corpora(ROOT / "data")
print("Discovered corpora:", list(corpora))

datasets = {}
for name, directory in corpora.items():
    faq = load_faq_dataset(directory)
    config = load_corpus_config(directory)
    datasets[name] = {"faq": faq, "config": config, "dir": directory}
    print(f"\n{config['display_name']}: {len(faq)} FAQs, "
          f"{faq['category'].nunique()} categories")
    print(f"  frozen preprocessing : {config.get('preprocessing_config')}")
    print(f"  frozen threshold     : {config['similarity_threshold']:.2f}")

In [ ]:
datasets["university"]["faq"].head(3)

### Where the records come from

Each record keeps the URL it came from. The university corpus deliberately
combines two source types, because the primary dataset alone could not supply
500 records that survived review. See `docs/DATA_SOURCES.md` for the full
review record.

In [ ]:
for name, bundle in datasets.items():
    print(name)
    print(bundle["faq"]["source_type"].value_counts().to_string(), "\n")

## 3. Preprocessing

`preprocess_text` lowercases, removes URLs and HTML, strips punctuation, and
tokenizes. Two switches are optional, and the project compares three
configurations built from them.

Note that negation words (`no`, `not`, `nor`) are **kept** even when stop words
are removed, because dropping them would reverse the meaning of a question.

In [ ]:
sample = "How do I <b>NOT</b> lose my classes at https://portal.example.edu?"

print("raw               :", sample)
print("basic             :", preprocess_text(sample))
print("stopwords removed :", preprocess_text(sample, remove_stopwords=True))
print("lemmatized        :", preprocess_text(sample, lemmatize=True))

In [ ]:
for name, options in PREPROCESSING_CONFIGS:
    print(f"{name:20} {options}")

## 4. Building the TF-IDF index

Only the **questions** are indexed. Answers never enter the similarity vectors,
so a long answer cannot pull an unrelated question to the top of the ranking.

The vectorizer is fitted once per corpus. A user query is then transformed with
the *same* fitted vocabulary, which is why the query and the corpus must share
one preprocessing configuration.

In [ ]:
for name, bundle in datasets.items():
    config = bundle["config"]
    options = {
        "remove_stopwords": bool(config["remove_stopwords"]),
        "lemmatize": bool(config["lemmatize"]),
    }
    vectorizer, matrix = build_tfidf_index(bundle["faq"], options)
    bundle["options"] = options
    bundle["vectorizer"] = vectorizer
    bundle["matrix"] = matrix
    print(f"{config['display_name']:18} matrix {matrix.shape} "
          f"vocabulary {len(vectorizer.vocabulary_)}")

In [ ]:
# Proof that answers are excluded: a word that appears only in answers is
# absent from the fitted vocabulary.
uni = datasets["university"]
vocab = set(uni["vectorizer"].vocabulary_)
question_words = set(preprocess_text(" ".join(uni["faq"]["question"])).split())
print("vocabulary size          :", len(vocab))
print("vocabulary minus questions:", len(vocab - question_words))

## 5. Retrieving an answer

`retrieve_tfidf` returns the top-k FAQs ranked by cosine similarity.
`answer_query` adds the threshold decision on top of that ranking.

In [ ]:
def ask(corpus_name, question, top_k=3):
    """Run one query against a corpus and print the outcome."""
    bundle = datasets[corpus_name]
    result = answer_query(
        question,
        bundle["faq"],
        bundle["vectorizer"],
        bundle["matrix"],
        threshold=float(bundle["config"]["similarity_threshold"]),
        top_k=top_k,
        preprocessing_options=bundle["options"],
    )
    print(f"Query    : {question}")
    print(f"Corpus   : {bundle['config']['display_name']} "
          f"(threshold {result['threshold']:.2f})")
    if result["found"]:
        best = result["best_match"]
        print(f"ACCEPTED : similarity {best['similarity']:.4f}")
        print(f"Matched  : {best['question']}")
        print(f"Answer   : {best['answer'][:300]}")
    else:
        top = result["top_matches"]
        score = top[0]["similarity"] if top else 0.0
        print(f"REJECTED : best similarity {score:.4f} is below the threshold")
    print("\nTop matches:")
    for rank, match in enumerate(result["top_matches"], start=1):
        print(f"  {rank}. {match['similarity']:.4f}  {match['question'][:80]}")
    print("-" * 78)
    return result

In [ ]:
# An exact question always scores 1.0 against itself.
exact = datasets["ecommerce"]["faq"]["question"].iloc[0]
_ = ask("ecommerce", exact)

In [ ]:
# A genuine paraphrase: different words, same intent.
_ = ask("ecommerce", "How do I send a product back and get my money returned?")

In [ ]:
# Out-of-domain question, correctly rejected by the e-commerce corpus.
_ = ask("ecommerce", "How do I train a puppy to sit?")

The *same* question is wrongly **accepted** by the university corpus. Its
threshold is lower (0.46 against 0.58), and the query shares the common words
*how*, *do*, *I*, *to* with a question about amending an application. This is a
real false acceptance, shown rather than hidden: it is exactly what the
answerable/unanswerable score overlap in section 8 predicts.

In [ ]:
_ = ask("university", "How do I train a puppy to sit?")

### Why the threshold matters

Without a threshold the system would always return its best guess, however poor.
The same out-of-domain query is shown below at threshold 0.0 and at the frozen
threshold, to make the rejection behaviour explicit.

In [ ]:
bundle = datasets["university"]
query = "What is the current price of Bitcoin?"
for threshold in [0.0, float(bundle["config"]["similarity_threshold"])]:
    result = answer_query(
        query, bundle["faq"], bundle["vectorizer"], bundle["matrix"],
        threshold=threshold, top_k=1,
        preprocessing_options=bundle["options"],
    )
    verdict = "ACCEPTED" if result["found"] else "REJECTED"
    score = result["top_matches"][0]["similarity"]
    print(f"threshold {threshold:.2f} -> {verdict} (best similarity {score:.4f})")

## 6. Choosing the preprocessing and the threshold

This is the part that must not touch the test set.

`tune_threshold` sweeps every threshold from 0.00 to 1.00 in 0.01 steps for each
of the three preprocessing configurations, using **validation queries only**.
Each threshold scores as the mean of two rates:

- the share of answerable queries that are correctly accepted
- the share of unanswerable queries that are correctly rejected

Ties are broken towards the **simpler** preprocessing configuration and then
towards the **higher** threshold, so the choice is reproducible rather than
arbitrary.

In [ ]:
tuning = {}
for name, bundle in datasets.items():
    validation = load_query_dataset(
        bundle["dir"] / "validation_queries.csv", set(bundle["faq"]["id"])
    )
    result = tune_threshold(validation, bundle["faq"])
    tuning[name] = result
    print(f"=== {bundle['config']['display_name']} "
          f"({len(validation)} validation queries) ===")
    for row in result["per_config_best"]:
        print(f"  {row['config']:20} threshold {row['threshold']:.2f}  "
              f"score {row['score']:.4f}  "
              f"accept {row['answerable_acceptance_rate']:.3f}  "
              f"reject {row['unanswerable_rejection_rate']:.3f}")
    print(f"  -> selected {result['selected_config']} at "
          f"{result['selected_threshold']:.2f}\n")

The two corpora choose differently, which is the whole point of comparing them.
Lemmatization helps the university corpus, whose questions vary in number and
tense. It does not help the e-commerce corpus, whose questions are dominated by
brand and product names that lemmatization leaves unchanged.

In [ ]:
# How the score varies with the threshold, for the selected configuration.
for name, result in tuning.items():
    sweep = pd.DataFrame(result["sweep"])
    best = sweep[sweep["config"] == result["selected_config"]]
    peak = best[best["score"] == best["score"].max()]
    print(f"{name}: best score {best['score'].max():.4f} is reached at "
          f"thresholds {peak['threshold'].min():.2f}-{peak['threshold'].max():.2f}; "
          f"the highest is chosen ({result['selected_threshold']:.2f})")

## 7. Final evaluation on the held-out test set

The configuration is now frozen. The test set is read once and never used to
adjust anything.

In [ ]:
reports = {}
for name, bundle in datasets.items():
    test = load_query_dataset(
        bundle["dir"] / "test_queries.csv", set(bundle["faq"]["id"])
    )
    report = evaluate_tfidf(
        test, bundle["faq"], bundle["vectorizer"], bundle["matrix"],
        threshold=float(bundle["config"]["similarity_threshold"]),
        preprocessing_options=bundle["options"],
    )
    reports[name] = report

summary = pd.DataFrame({
    datasets[name]["config"]["display_name"]: {
        "Top-1 accuracy": round(r["top1_accuracy"], 3),
        "Top-3 accuracy": round(r["top3_accuracy"], 3),
        "Mean similarity of correct Top-1": round(r["mean_similarity_correct_top1"], 4),
        "Answerable acceptance rate": round(r["answerable_acceptance_rate"], 3),
        "Unanswerable rejection rate": round(r["unanswerable_rejection_rate"], 3),
        "False acceptances": r["false_acceptance_count"],
        "False rejections": r["false_rejection_count"],
    }
    for name, r in reports.items()
})
summary

### Reading these numbers

- **Top-3 is far above Top-1 on the university corpus** (0.987 vs 0.847). The
  right FAQ is nearly always retrieved; it is the *ranking* that slips, usually
  between two closely related programmes.
- **E-commerce rejects 26 answerable queries.** Its threshold is high (0.58)
  because its unanswerable queries score highly, so genuine paraphrases that
  reword too much fall below the line. This is the precision/recall trade-off
  the threshold controls.
- **False acceptance and false rejection are threshold errors**, counted against
  the gold `is_answerable` label. An answerable query that is accepted but shows
  the wrong FAQ is reported separately, below.

## 8. Error analysis

These are the cases the teacher is most likely to ask about, so they are shown
rather than summarised away.

In [ ]:
for name, report in reports.items():
    title = datasets[name]["config"]["display_name"]
    print(f"=== {title}: incorrect Top-1 retrievals ===")
    for example in report["incorrect_retrieval_examples"][:3]:
        print(f"  Query     : {example['query'][:95]}")
        print(f"  Expected  : FAQ {example['expected_faq_id']}")
        print(f"  Retrieved : FAQ {example['retrieved_faq_id']} - "
              f"{example['retrieved_question'][:70]}")
        print(f"  Similarity: {example['similarity']:.4f}  "
              f"accepted={example['accepted']}\n")

In [ ]:
# Where do answerable and unanswerable queries actually sit on the score scale?
for name, bundle in datasets.items():
    test = load_query_dataset(
        bundle["dir"] / "test_queries.csv", set(bundle["faq"]["id"])
    )
    scores = []
    for query in test["query"]:
        matches = retrieve_tfidf(
            query, bundle["faq"], bundle["vectorizer"], bundle["matrix"],
            top_k=1, preprocessing_options=bundle["options"],
        )
        scores.append(matches[0]["similarity"] if matches else 0.0)
    test = test.assign(top_score=scores)
    answerable = test[test["is_answerable"]]["top_score"]
    unanswerable = test[~test["is_answerable"]]["top_score"]
    print(f"{bundle['config']['display_name']} "
          f"(threshold {bundle['config']['similarity_threshold']:.2f})")
    print(f"  answerable   mean {answerable.mean():.3f}  "
          f"min {answerable.min():.3f}  max {answerable.max():.3f}")
    print(f"  unanswerable mean {unanswerable.mean():.3f}  "
          f"min {unanswerable.min():.3f}  max {unanswerable.max():.3f}")
    print(f"  the two ranges overlap, which is why no threshold scores 1.000\n")

That overlap is the honest headline result: **a single cosine threshold cannot
perfectly separate answerable from unanswerable queries**, because a short
out-of-domain question can share common words with a short FAQ question. This is
a limitation of bag-of-words TF-IDF, and it is what later phases using
Word2Vec or Transformer embeddings are meant to improve.

## 9. Ask your own question

Change the corpus and the question below and re-run the cell.

In [ ]:
ask("university", "What are the entry criteria for a graduate programme?")
ask("ecommerce", "My package has not arrived yet, what should I do?")
None

## 10. Summary

| | University FAQ | E-commerce FAQ |
| --- | --- | --- |
| FAQs indexed | 500 | 500 |
| Preprocessing (validation-selected) | lemmatized | basic |
| Threshold (validation-selected) | 0.46 | 0.58 |
| Top-1 accuracy | 0.847 | 0.953 |
| Top-3 accuracy | 0.987 | 0.980 |

**What was built:** a class-free, corpus-configurable TF-IDF retrieval engine
with schema validation, three comparable preprocessing configurations, a
reproducible threshold-selection protocol that never touches the test set, and
34 unit tests.

**What it cannot do:** match meaning without shared vocabulary, or separate
answerable from unanswerable queries perfectly with one threshold.

**Honest caveat on the numbers:** the evaluation queries are generated
paraphrases, not human-written ones. They are measurably distinct from their
sources — no query reproduces its source question, and mean source-token overlap
is 0.59 (university) and 0.77 (e-commerce) — but the reported accuracy should be
read as an estimate over synthetic paraphrases. `docs/DATA_SOURCES.md` records
how they were produced and measured.